# 03 — Baselines: LLM judges (open models, 4-bit, free Colab T4)

**What this notebook does.** Two open instruction-tuned LLMs act as judges. For each (question, reference, candidate) the judge rates the candidate from 1 to 5 against the reference, using the rubric in `src/common.py`.

**Two prompt variants per judge**
* `plain`: the rubric only. This is the standard LLM-as-judge setup and the one that can show register bias.
* `agnostic`: the same prompt plus an explicit instruction to ignore dialect, script and style. This is the cheapest possible mitigation, so it is the baseline our trained judge must beat.

**How the score is read (important).** We do **not** parse generated text. We take the model's logits for the five digit tokens `1..5` at the answer position, apply a softmax over those five, and use the **expected score** `sum(p_i * i)`. Advantages: deterministic (no sampling), no parsing failures, and a continuous score, so a small systematic penalty (4.6 vs 4.2) is visible instead of being rounded away. The probabilities are saved too (`results/llm_probs/`).

**Resources.** Models are loaded in 4-bit NF4 (about 5-6 GB each), one at a time, and freed before the next. The Llama model is gated: accept its licence on Hugging Face and add an `HF_TOKEN` secret; if that is not possible, replace it in `JUDGES` with another non-gated instruct model (for example `Qwen/Qwen2.5-7B-Instruct`). No paid API is used anywhere.

In [ ]:
import os, sys
REPO_DIR = "register_bias_judge"
if not os.path.exists("src/common.py"):
    if os.path.isdir(REPO_DIR):                                   # unzipped next to the notebook
        os.chdir(REPO_DIR)
    else:
        try:                                                      # Google Colab: the repo lives in your Drive, so
            from google.colab import drive                        # data/ and results/ survive between sessions
            drive.mount("/content/drive")
            os.chdir(f"/content/drive/MyDrive/{REPO_DIR}")
        except Exception as e:
            print("Could not locate the repo folder:", e)
sys.path.insert(0, os.getcwd())
assert os.path.exists("src/common.py"), "Folder 'register_bias_judge' not found. Put it in the root of your Google Drive (My Drive) and re-run."
print("working in:", os.getcwd())

In [ ]:
%pip install -q "transformers>=4.51" accelerate bitsandbytes scipy

In [ ]:
import numpy as np, pandas as pd, torch
from src import common as C
from src import models as M

C.set_seed(C.SEED)
FOUR_BIT = torch.cuda.is_available()      # True on Colab/Kaggle GPU; falls back to full precision on CPU for testing
JUDGES = {                                 # name -> Hugging Face id (verify these are still the current releases)
    "qwen3-8b":    "Qwen/Qwen3-8B",
    "llama3.1-8b": "meta-llama/Llama-3.1-8B-Instruct",
}
VARIANTS = ["plain", "agnostic"]
BATCH = 8
M.hf_login()
(C.RESULTS / "llm_probs").mkdir(exist_ok=True)

In [ ]:
items = C.load_items()
pairs = C.build_pairs(items)
print(pairs.shape)
print("\nExample prompt (agnostic variant):\n")
ex = pairs.iloc[0]
print(C.judge_messages(ex.question, ex.reference, ex.candidate, "agnostic")[0]["content"])

## Run every judge x prompt variant over all pairs

In [ ]:
records = []
for jname, hf_id in JUDGES.items():
    try:
        model, tok = M.load_causal_lm(hf_id, four_bit=FOUR_BIT)
        print(f"{jname}: loaded | digit token ids = {M.digit_token_ids(tok)}")
        for variant in VARIANTS:
            ev = f"{jname}|{variant}"
            s, probs = M.judge_pairs(model, tok, pairs, variant, batch_size=BATCH, desc=ev)
            records.append(pd.DataFrame({"pair_id": pairs.pair_id, "evaluator": ev, "family": "llm", "score": s}))
            pd.concat([pairs[["pair_id"]], pd.DataFrame(probs, columns=[f"p{d}" for d in M.DIGITS])], axis=1) \
              .to_csv(C.RESULTS / "llm_probs" / f"{jname}_{variant}.csv", index=False)
            print(f"  done: {ev}")
        del model
        M.free_gpu()
    except Exception as e:
        print(f"[skipped] {jname}: {type(e).__name__}: {str(e)[:300]}")
        M.free_gpu()

## Save and sanity-check

Same check as notebook 02: a judge must rank correct > partial > incorrect. We also print the average expected score per level.

In [ ]:
scores = pd.concat(records, ignore_index=True)
print("saved:", C.save_scores(scores, "llm"))
chk = scores.merge(pairs[["pair_id", "level"]], on="pair_id")
tab = chk.groupby(["evaluator", "level"]).score.mean().unstack()[C.LEVELS]
tab["separates_levels"] = (tab.correct > tab.partial) & (tab.partial > tab.incorrect)
tab.round(2)